In [1]:
import pandas as pd
from pathlib import Path

In [2]:
def leer_paro(ruta):
    df = pd.read_csv(
        ruta,
        encoding='latin-1',
        sep=';',
        skiprows=1
    )
    return df

Limpiar valores <5 y convertir a numérico.

El INE oculta valores muy pequeños (no dice "hay 3 parados" en un pueblo de 10 habitantes para no identificar a nadie). En su lugar pone <5. Como no podemos dejarlo como texto, lo aproximamos a 2 (punto medio entre 0 y 5).

In [3]:
def limpiar_total_paro(df):
    df['total Paro Registrado'] = (
        df['total Paro Registrado']
        .astype(str)
        .str.replace('<5', '2', regex=False)
        .str.replace('.', '', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    )
    return df

Agregar media anual a las CCAA

In [4]:
def agregar_media_anual(df, año):
    df_mensual = df.groupby(
        ['Comunidad Autónoma', 'mes'], as_index=False
    )['total Paro Registrado'].sum()

    df_anual = df_mensual.groupby(
        'Comunidad Autónoma', as_index=False
    )['total Paro Registrado'].mean()

    df_anual['total Paro Registrado'] = df_anual['total Paro Registrado'].round(0).astype(int)

    df_anual['año'] = año
    df_anual = df_anual.rename(columns={
        'Comunidad Autónoma': 'comunidad',
        'total Paro Registrado': 'paro_medio'
    })

    return df_anual[['comunidad', 'año', 'paro_medio']]

In [5]:
def limpiar_nombre_comunidad(df):
    df['comunidad'] = (
        df['comunidad']
        .str.replace('Asturias, Principado de', 'Asturias', regex=False)
        .str.replace('Balears, Illes', 'Baleares', regex=False)
        .str.replace('Castilla - La Mancha', 'Castilla-La Mancha', regex=False)
        .str.replace('Comunitat Valenciana', 'Comunidad Valenciana', regex=False)
        .str.replace('Madrid, Comunidad de', 'Madrid', regex=False)
        .str.replace('Murcia, Región de', 'Murcia', regex=False)
        .str.replace('Navarra, Comunidad Foral de', 'Navarra', regex=False)
        .str.replace('Rioja, La', 'La Rioja', regex=False)
    )
    return df

In [6]:
def procesar_archivo_paro(ruta, año):
    df = leer_paro(ruta)
    df = limpiar_total_paro(df)
    df = agregar_media_anual(df, año)
    df = limpiar_nombre_comunidad(df)
    return df

In [7]:
def main():
    ruta_raw = Path(
        r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\01_raw\indicadores sociales\desempleo"
    )

    ruta_salida = Path(
        r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\02_processed\indicadores_sociales\desempleo"
    )
    ruta_salida.mkdir(exist_ok=True)
    
    años = range(2014, 2025)  # 2014 a 2024 incluidos
    
    resultados = []
    for año in años:
        ruta_archivo = ruta_raw / f"Paro_por_municipios_{año}_csv.csv"
        df_año = procesar_archivo_paro(ruta_archivo, año)
        resultados.append(df_año)
    
    desempleo_total = pd.concat(resultados, ignore_index=True)
    desempleo_total.to_csv(
        ruta_salida / "desempleo_españa.csv",
        index=False,
        encoding='utf-8-sig'
    )
    
    return desempleo_total


desempleo_total = main()
print(desempleo_total.shape)
print(sorted(desempleo_total['comunidad'].unique()))

(209, 3)
['Andalucía', 'Aragón', 'Asturias', 'Baleares', 'Canarias', 'Cantabria', 'Castilla y León', 'Castilla-La Mancha', 'Cataluña', 'Ceuta', 'Comunidad Valenciana', 'Extremadura', 'Galicia', 'La Rioja', 'Madrid', 'Melilla', 'Murcia', 'Navarra', 'País Vasco']
